In [ ]:
# 1. the kernel started: machine, disk, mounts
import os, subprocess, time, sys
T0 = time.time()
def sh(c):
    print("$", c, flush=True)
    print(subprocess.run(c, shell=True, capture_output=True, text=True).stdout[-6000:], flush=True)
print("kernel alive", time.strftime("%H:%M:%S"), flush=True)
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv")
sh("free -g; nproc; df -h / /kaggle/working /kaggle/input /tmp 2>/dev/null; ldd --version | head -1")
sh("find /kaggle/input -maxdepth 6 -type f -size +1M -printf '%s %p\n' | sort -k2 | head -60")
sh("mount | grep -i kaggle/input | head")

In [ ]:
# 2. stage the llama.cpp folder, try both ways of starting it
import shutil
from pathlib import Path
srv = sorted(Path("/kaggle/input").rglob("llama-server"))
print("llama-server at", srv, flush=True)
RT = Path("/tmp/llamacpp")
shutil.rmtree(RT, ignore_errors=True)
shutil.copytree(srv[0].parent, RT)
for p in RT.iterdir():
    p.chmod(0o755)
env = os.environ.copy()
env["LD_LIBRARY_PATH"] = f"{RT}:" + env.get("LD_LIBRARY_PATH", "")
print("host LD_LIBRARY_PATH", os.environ.get("LD_LIBRARY_PATH"), flush=True)
LAUNCH = None
for how, cmd in (("direct", [str(RT / "llama-server")]),
                 ("loader", [str(RT / "ld-linux-x86-64.so.2"), "--library-path", env["LD_LIBRARY_PATH"], str(RT / "llama-server")])):
    r = subprocess.run(cmd + ["--version"], env=env, capture_output=True, text=True, timeout=120)
    print(how, "rc", r.returncode, (r.stdout + r.stderr)[-2000:], flush=True)
    if r.returncode == 0 and LAUNCH is None:
        LAUNCH = cmd
print("LAUNCH", LAUNCH, flush=True)

In [ ]:
# 3. start the server on the model (vision on), wait until it listens, one request, stop
import json, urllib.request
gg = sorted(Path("/kaggle/input").rglob("*-00001-of-*.gguf"))
mm = sorted(Path("/kaggle/input").rglob("mmproj*.gguf"))
print("model", gg, "mmproj", mm, flush=True)
log = open("/kaggle/working/llama-server.log", "w")
cmd = LAUNCH + ["--model", str(gg[0]), "--mmproj", str(mm[0]), "--alias", "glm", "--host", "127.0.0.1", "--port", "8000",
                "--parallel", "10", "--kv-unified", "--kv-unified-per-slot", "131072", "--fit", "on", "--fit-target", "3072",
                "--batch-size", "4096", "--ubatch-size", "2048", "--flash-attn", "auto", "--jinja",
                "--reasoning-format", "deepseek", "--cache-ram", "16384", "--ctx-checkpoints", "32", "--metrics", "--no-warmup"]
print(" ".join(cmd), flush=True)
t = time.time()
proc = subprocess.Popen(cmd, env=env, stdout=log, stderr=subprocess.STDOUT)
up = False
while time.time() - t < 2400 and proc.poll() is None:
    try:
        up = urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=5).status == 200
    except Exception:
        up = False
    if up:
        break
    time.sleep(15)
    if int(time.time() - t) % 120 < 15:
        print(f"{time.time() - t:.0f} s", open("/kaggle/working/llama-server.log").read().splitlines()[-1:], flush=True)
        sh("free -g | head -2; nvidia-smi --query-gpu=memory.used --format=csv,noheader")
print("up", up, "after", round(time.time() - t), "s, rc", proc.poll(), flush=True)
if up:
    for effort in ("low", "max"):
        body = {"model": "glm", "messages": [{"role": "user", "content": "What is 17*23? Answer with the number."}],
                "max_tokens": 2048, "temperature": 1.0, "top_p": 0.95, "chat_template_kwargs": {"reasoning_effort": effort}}
        t1 = time.time()
        req = urllib.request.Request("http://127.0.0.1:8000/v1/chat/completions", json.dumps(body).encode(),
                                     {"Content-Type": "application/json"})
        r = json.loads(urllib.request.urlopen(req, timeout=1800).read())
        print(effort, round(time.time() - t1, 1), "s", json.dumps(r)[:3000], flush=True)
    sh("free -g | head -2; nvidia-smi --query-gpu=memory.used --format=csv,noheader")
proc.terminate()
time.sleep(5)
log.close()
L = open("/kaggle/working/llama-server.log").read().splitlines()
print("\n".join(L[:150]), "\n...\n", "\n".join(L[-120:]), flush=True)
print("total", round(time.time() - T0), "s")